In [ ]:
import time
from os import times

from attr import dataclass
from playwright.async_api import async_playwright

playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=False)
page = await browser.new_page()

In [ ]:
from io import BytesIO
from PIL import Image
import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim

CHANGE_THRESHOLD = 20  # per-channel difference we count as "this pixel moved"


def decode_pair(img1_bytes, img2_bytes):
    a = np.array(Image.open(BytesIO(img1_bytes)).convert("RGB"))
    b = np.array(Image.open(BytesIO(img2_bytes)).convert("RGB"))
    return a, b


def screenshot_similarity(img1_bytes, img2_bytes):
    a, b = decode_pair(img1_bytes, img2_bytes)

    score = ssim(
        a,
        b,
        channel_axis=2,
        data_range=255,
    )
    return score


def mean_abs_diff(img1_bytes, img2_bytes):
    """Average absolute difference per channel, 0-255.

    Same family as the old mse but L1 instead of L2, so a single bright
    region can no longer dominate the whole score by being squared.
    """
    a, b = decode_pair(img1_bytes, img2_bytes)
    return float(cv2.absdiff(a, b).mean())


def changed_pixel_ratio(img1_bytes, img2_bytes, threshold=CHANGE_THRESHOLD):
    """Fraction of pixels that moved by more than `threshold` on any channel."""
    a, b = decode_pair(img1_bytes, img2_bytes)
    diff = cv2.absdiff(a, b)
    changed = np.any(diff > threshold, axis=2)
    return changed.mean()


def show_screenshot(screenshot):
    return Image.open(BytesIO(screenshot)).convert("RGB")


def show_image_sizes(images):
    count = 1
    for img in images:
        print(f"#{count} - size: {len(img)}")
        count += 1

In [ ]:
from dataclasses import dataclass


@dataclass
class SampleData:
    screenshot1: bytes
    screenshot2: bytes
    name: str
    # idle=True  -> nothing was clicked/typed/navigated between the two shots.
    #               Any difference is the page changing on its own (clocks,
    #               live prices, carousels, ads, animation).
    # idle=False -> a user action happened between the two shots.
    idle: bool

In [ ]:
data_pairs = []

In [ ]:
# Sample #1
import time

await page.goto("http://localhost:8001/shop/product.html")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("//*[@id='add-to-cart']").click()
time.sleep(2)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="e-commerce", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #2
import time

await page.goto("https://time.is/")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
time.sleep(2)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
show_image_sizes([screenshot1, screenshot2])
data = SampleData(screenshot1, screenshot2, name="time.is", idle=True)
data_pairs.append(data)

In [ ]:
abs(len(screenshot1) - len(screenshot2))

In [ ]:
# Sample #3
import time

await page.goto("https://news.ycombinator.com/newest")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
time.sleep(2)
await page.goto("https://news.ycombinator.com/front")
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="hackernews", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #4
import time

await page.goto("https://coinmarketcap.com/currencies/bitcoin/")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
time.sleep(5)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="coinmarketcap", idle=True)
data_pairs.append(data)

In [ ]:
show_screenshot(screenshot2)

In [ ]:
# Sample #5 - idle page, no visual change at all (baseline)
import time

await page.goto("https://en.wikipedia.org/wiki/Web_browser")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
time.sleep(2)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="wikipedia-idle", idle=True)
data_pairs.append(data)

In [ ]:
# Sample #6 - focusing search opens a large overlay (big change from one click)
import time

await page.goto("https://en.wikipedia.org/wiki/Main_Page")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("input[name='search']").first.click()
await page.keyboard.type("playwright")
time.sleep(2)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="wikipedia-search-suggestions", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #7 - same page, scrolled down (whole viewport changes, same document)
import time

await page.goto("https://en.wikipedia.org/wiki/Web_browser")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.mouse.wheel(0, 1200)
time.sleep(2)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="wikipedia-scroll", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #8 - navigation to a completely different site
import time

await page.goto("https://example.com/")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.goto("https://www.iana.org/help/example-domains")
time.sleep(1)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="cross-site-navigation", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #9 - async content finishes loading after a click (spinner -> text)
import time

await page.goto("https://the-internet.herokuapp.com/dynamic_loading/1")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("#start button").click()
time.sleep(6)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="dynamic-loading", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #10 - tiny change: a single checkbox toggled
import time

await page.goto("https://the-internet.herokuapp.com/checkboxes")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("#checkboxes input").first.check()
time.sleep(1)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="checkbox-toggle", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #11 - typing into a login form (text appears inside two inputs)
import time

await page.goto("https://the-internet.herokuapp.com/login")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("#username").fill("tomsmith")
await page.locator("#password").fill("SuperSecretPassword!")
time.sleep(1)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="login-form-fill", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #12 - successful form submit -> flash message + new page
import time

await page.goto("https://the-internet.herokuapp.com/login")
await page.locator("#username").fill("tomsmith")
await page.locator("#password").fill("SuperSecretPassword!")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("button[type='submit']").click()
time.sleep(3)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="login-submit-success", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #13 - infinite scroll appends more content
import time

await page.goto("https://the-internet.herokuapp.com/infinite_scroll")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.keyboard.press("End")
time.sleep(3)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="infinite-scroll", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #14 - hover reveals a caption overlay (very small localized change)
import time

await page.goto("https://the-internet.herokuapp.com/hovers")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator(".figure").first.hover()
time.sleep(1)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="hover-reveal-caption", idle=False)
data_pairs.append(data)

In [ ]:
# Helper for idle samples: no interaction at all, just wait and shoot again.
# Everything that differs between the two shots changed without us touching it.
import time


async def collect_idle(url, name, settle=2, wait=5):
    await page.goto(url, wait_until="domcontentloaded")
    time.sleep(settle)  # let the page settle before the first shot
    screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
    time.sleep(wait)
    screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
    data_pairs.append(SampleData(screenshot1, screenshot2, name=name, idle=True))
    print(f"collected idle sample: {name}")

In [ ]:
# Idle samples #15-#27 - public pages, no user action between the two shots.
# The spread is deliberate: from a page that never moves to one that repaints
# every frame.
IDLE_SITES = [
    ("example-static", "https://example.com/"),
    ("nasa-home", "https://www.nasa.gov/"),
    ("bbc-news", "https://www.bbc.com/news"),
    ("openweathermap-london", "https://openweathermap.org/city/5128581"),
    ("coinbase-btc-price", "https://www.coinbase.com/price/bitcoin"),
    ("timeanddate-worldclock", "https://www.timeanddate.com/worldclock/"),
    ("flightradar24", "https://www.flightradar24.com/"),
    ("clock-zone", "https://clock.zone/"),
    ("wikipedia-recent-changes", "https://en.wikipedia.org/wiki/Special:RecentChanges"),
    ("time-gov", "https://www.time.gov/"),
    ("yahoo-finance-btc", "https://finance.yahoo.com/quote/BTC-USD/"),
    ("espn-home", "https://www.espn.com/"),
    ("webgl-aquarium", "https://webglsamples.org/aquarium/aquarium.html"),
]

for name, url in IDLE_SITES:
    try:
        await collect_idle(url, name)
    except Exception as e:
        print(f"skipped {name}: {type(e).__name__}: {e}")

In [ ]:
# Sample #28 - GitHub: switch to the Issues tab
import time

await page.goto("https://github.com/microsoft/playwright", wait_until="domcontentloaded")
time.sleep(2)
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.get_by_role("link", name="Issues").first.click()
time.sleep(3)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="github-issues-tab", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #29 - Hacker News: open the comments page of the top story
import time

await page.goto("https://news.ycombinator.com/", wait_until="domcontentloaded")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("a:has-text('comments')").first.click()
time.sleep(3)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="hn-open-comments", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #30 - DuckDuckGo: submit a search, blank form -> results page
import time

await page.goto("https://html.duckduckgo.com/html/", wait_until="domcontentloaded")
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("input[name='q']").first.fill("playwright python")
await page.keyboard.press("Enter")
time.sleep(3)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="duckduckgo-search", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #31 - python.org: search submit, homepage -> results
import time

await page.goto("https://www.python.org/", wait_until="domcontentloaded")
time.sleep(2)
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("#id-search-field").fill("asyncio")
await page.keyboard.press("Enter")
time.sleep(4)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="python-org-search", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #32 - MDN: open the theme menu (small dropdown over a static page)
import time

await page.goto("https://developer.mozilla.org/en-US/docs/Web/HTML", wait_until="domcontentloaded")
time.sleep(2)
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.get_by_role("button", name="Theme").first.click()
time.sleep(2)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="mdn-theme-menu", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #33 - Wikipedia portal: type-ahead suggestions under the search box
import time

await page.goto("https://www.wikipedia.org/", wait_until="domcontentloaded")
time.sleep(1)
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.locator("#searchInput").fill("screenshot")
time.sleep(2)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="wikipedia-portal-typeahead", idle=False)
data_pairs.append(data)

In [ ]:
# Sample #34 - wikiHow: scroll a long content page
import time

await page.goto("https://www.wikihow.com/Main-Page", wait_until="domcontentloaded")
time.sleep(2)
screenshot1 = await page.screenshot(full_page=False, type="webp", quality=40)
await page.mouse.wheel(0, 1500)
time.sleep(2)
screenshot2 = await page.screenshot(full_page=False, type="webp", quality=40)
data = SampleData(screenshot1, screenshot2, name="wikihow-scroll", idle=False)
data_pairs.append(data)

In [ ]:
# Per-sample metrics, split by idle vs action-driven.
results = []

for data in data_pairs:
    score = screenshot_similarity(data.screenshot1, data.screenshot2)
    abs_diff = mean_abs_diff(data.screenshot1, data.screenshot2)
    pixel_change_ratio = changed_pixel_ratio(data.screenshot1, data.screenshot2)
    byte_delta = abs(len(data.screenshot1) - len(data.screenshot2))
    results.append((data.name, data.idle, score, abs_diff, pixel_change_ratio, byte_delta))

header = f"{'sample':30s} {'kind':7s} {'ssim':>8s} {'abs_diff':>9s} {'pix_ratio':>10s} {'d_bytes':>9s}"
print(header)
print("-" * len(header))
for name, idle, score, abs_diff, ratio, dbytes in results:
    kind = "idle" if idle else "action"
    print(f"{name:30s} {kind:7s} {score:8.4f} {abs_diff:9.3f} {ratio:10.5f} {dbytes:9d}")

In [ ]:
# Idle vs action-driven: can a single threshold tell them apart?
def describe(rows, label):
    if not rows:
        print(f"{label}: no samples")
        return
    ssims = sorted(r[2] for r in rows)
    diffs = sorted(r[3] for r in rows)
    ratios = sorted(r[4] for r in rows)
    mid = len(ssims) // 2
    print(f"{label} (n={len(rows)})")
    print(f"  ssim      min={ssims[0]:.4f}  median={ssims[mid]:.4f}  max={ssims[-1]:.4f}")
    print(f"  abs_diff  min={diffs[0]:.3f}  median={diffs[mid]:.3f}  max={diffs[-1]:.3f}")
    print(f"  pix_ratio min={ratios[0]:.5f}  median={ratios[mid]:.5f}  max={ratios[-1]:.5f}")


idle_rows = [r for r in results if r[1]]
action_rows = [r for r in results if not r[1]]
describe(idle_rows, "idle")
print()
describe(action_rows, "action-driven")

# The overlap is where a naive threshold misfires: idle pages that move more
# than the quietest real action did.
if action_rows:
    floor = min(r[4] for r in action_rows)
    noisy = sorted([r for r in idle_rows if r[4] > floor], key=lambda r: -r[4])
    print(f"\nIdle pages noisier than the quietest action (pix_ratio {floor:.5f}):")
    for r in noisy:
        print(f"  {r[0]:30s} pix_ratio={r[4]:.5f}")
    if not noisy:
        print("  none")

In [ ]:
await browser.close()
await playwright.stop()
